# Information Retrieval and Web Analysis — Final Project Part 3
## Ranking & Filtering

**Serena El-Hajj · Souad Ibrahim El Husseini · Guillermo**  
**GitHub:** https://github.com/Souad-El-Husseini/irwa-search-engine


This notebook runs from our GitHub repository in Google Colab.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

repo_url = "https://github.com/Souad-El-Husseini/irwa-search-engine.git"
repo_name = "irwa-search-engine"

if "google.colab" in sys.modules:
    repo_root = Path("/content") / repo_name
    if not (repo_root / "project_progress" / "part_2" / "search_engine.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(repo_root)], check=True)
else:
    # Also works when the repository is opened online through GitHub Codespaces.
    current = Path.cwd().resolve()
    repo_root = next(
        (folder for folder in [current, *current.parents]
         if (folder / "project_progress" / "part_2" / "search_engine.py").exists()),
        None,
    )
    if repo_root is None:
        raise FileNotFoundError("Open from GitHub in Google Colab or in a repository Codespace")

# Install missing course dependencies in this cloud session.
packages = {"pandas": "pandas", "numpy": "numpy", "nltk": "nltk", "gensim": "gensim>=4.3.3"}
missing = [pip_name for module, pip_name in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import nltk
nltk.download("stopwords", quiet=True)
import pandas as pd
from IPython.display import display

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print("Using GitHub repository:", repo_root)


In [ ]:
from project_progress.part_3.part3_ranking import Part3Ranker, QUERIES

# Reuse the Part 2 data and indexing functions from this same repository.
engine = Part3Ranker.from_repo(repo_root)
print("Number of products:", engine.N)
print("Index vocabulary:", len(engine.df))
print("Indexed fields:", ", ".join(engine.token_fields[next(iter(engine.token_fields))].keys()))

## 1. AND retrieval

We reuse Part 2's AND search for all ranking methods.


In [ ]:
# We preserved the retrieval approach implemented in Part 2 
# and verified that the five queries returned the same matching products before applying the new ranking algorithms.

counts = pd.DataFrame([
    {"Query": query, "AND matches": len(engine.candidates(query))}
    for query in QUERIES
])
display(counts)


## 2. TF-IDF + cosine similarity

We rank the Part 2 matches using cosine-normalized TF-IDF.


In [ ]:
query = QUERIES[0]
print("TF-IDF cosine:", query)
display(engine.results_table(query, "TF-IDF cosine", top_k=10))


## 3. BM25

We use BM25 with `k1=1.5` and `b=0.75` on the same AND matches.


In [ ]:
print("BM25:", query)
display(engine.results_table(query, "BM25", top_k=10))


## 4. Custom Fashion Shopping Score

Our score prioritizes text similarity while considering title matches, ratings, discounts, and stock.

$$S(q,d)=0.60C(q,d)+0.15T(q,d)+0.10R(d)+0.05D(d)+0.10A(d)$$


In [ ]:
print("Custom Fashion Shopping Score:", query)
display(engine.results_table(query, "Custom", top_k=10))


## 5. Ranking comparison

We compare the top 10 products for all five queries and count the ranking changes.


In [ ]:
# Compare the top 10 rankings for all five queries.
comparison_summary = []
for q in QUERIES:
    rankings = {
        "TF-IDF cosine": [pid for pid, score in engine.tfidf_cosine(q)[:10]],
        "BM25": [pid for pid, score in engine.bm25(q)[:10]],
        "Custom": [pid for pid, score in engine.custom(q)[:10]],
    }
    comparison = pd.DataFrame({name: pd.Series(ids, index=range(1, len(ids) + 1))
                               for name, ids in rankings.items()})
    comparison.index.name = "Rank"
    print(f"\nQuery: {q}")
    display(comparison)
    original = rankings["TF-IDF cosine"]
    for method in ("BM25", "Custom"):
        other_ranks = {pid: rank for rank, pid in enumerate(rankings[method], 1)}
        shared = [pid for pid in original if pid in other_ranks]
        moved = sum(rank != other_ranks[pid]
                    for rank, pid in enumerate(original, 1) if pid in other_ranks)
        comparison_summary.append({"query": q, "compared_with": method,
                                   "shared_top10": len(shared),
                                   "shared_products_changing_rank": moved})

ranking_comparison_summary = pd.DataFrame(comparison_summary)
print("\nRanking comparison summary:")
display(ranking_comparison_summary)


## 6. Word2Vec + cosine similarity

We train Word2Vec on our seven preprocessed fields and rank the AND matches by cosine similarity of averaged word vectors.


In [ ]:
# Run once: learning Word2Vec on the 28,080-product corpus.
model = engine.train_word2vec(vector_size=100, window=5, epochs=10, seed=42)
print("Word2Vec vocabulary:", len(model.wv))
print("Word vector size:", model.vector_size)


## 7. Word2Vec top 20

We display the top 20 results for each Part 2 query, where available.


In [ ]:
word2vec_tables = []
for query_id, query in enumerate(QUERIES, 1):
    table = engine.results_table(query, "Word2Vec cosine", top_k=20)
    table.insert(0, "query", query)
    table.insert(0, "query_id", query_id)
    word2vec_tables.append(table)
    print(f"\nQUERY {query_id}: {query} | {len(engine.candidates(query))} AND matches")
    display(table)


## 8. Export results

We save rankings and run details to `results/part3/`.


In [ ]:
import json

output = repo_root / "results" / "part3"
output.mkdir(parents=True, exist_ok=True)
rows = []
for qid, q in enumerate(QUERIES, 1):
    for method in ["TF-IDF cosine", "BM25", "Custom", "Word2Vec cosine"]:
        tab = engine.results_table(q, method, top_k=20)
        tab.insert(0, "ranking_method", method)
        tab.insert(0, "query", q)
        tab.insert(0, "query_id", qid)
        rows.append(tab)
all_top20 = pd.concat(rows, ignore_index=True)
all_top20.to_csv(output / "part3_top20_all_methods.csv", index=False)
all_top20[all_top20.ranking_method == "Word2Vec cosine"].to_csv(
    output / "word2vec_top20_five_queries.csv", index=False)
counts.rename(columns={"Query": "query", "AND matches": "and_matches"}).to_csv(
    output / "query_counts.csv", index=False)
model.save(str(output / "word2vec.model"))
(output / "methodology.json").write_text(json.dumps({
    "corpus_size": engine.N, "vocabulary_size": len(engine.df),
    "query_semantics": "Part 2 Boolean AND",
    "methods": ["TF-IDF cosine", "BM25", "Custom", "Word2Vec cosine"],
    "bm25": {"k1": 1.5, "b": 0.75},
    "custom_weights": [0.60, 0.15, 0.10, 0.05, 0.10],
    "word2vec": {"size": 100, "window": 5, "epochs": 10, "seed": 42}
}, indent=2), encoding="utf-8")
print("Results saved to:", output)


## 9. Alternative text representations

Doc2Vec learns document embeddings directly. Pretrained sentence embeddings could capture context better than averaged Word2Vec, but use more resources. We discuss the trade-offs in our report.


## 10. Download results (Colab)

The ZIP contains our Part 3 outputs for upload to GitHub.


In [ ]:
import shutil

results_folder = repo_root / "results" / "part3"
zip_path = shutil.make_archive("/content/irwa_part3_results" if "google.colab" in sys.modules
                               else str(repo_root / "irwa_part3_results"),
                               "zip", root_dir=results_folder)
print("Created:", zip_path)

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(zip_path)
else:
    print("Open the ZIP in the current cloud workspace and add its outputs to GitHub.")